# 🎮 게임 STT 데이터 기반 6개 모델 벤치마크

- **데이터**: `game_test.tsv` (187건)
- **목적**: 게임 내 혐오 표현(욕설, 비난 등) 탐지 성능 비교
- **모델**: 6개 감정/혐오 분석 모델 테스트

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import gc
import json
import os
from datetime import datetime
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

# 폰트 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정
DATA_PATH = "data/game_test.tsv"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1. 모델 및 데이터 설정

In [ ]:
# 테스트할 모델 목록
MODELS_TO_TEST = [
    ("Korean Sentiment", "matthewburke/korean_sentiment"),
    ("KoELECTRA Small", "monologg/koelectra-small-finetuned-sentiment"),
    ("KoELECTRA Base", "monologg/koelectra-base-finetuned-sentiment"),
    ("Multilingual", "nlptown/bert-base-multilingual-uncased-sentiment"),
    ("UnSmile", "smilegate-ai/kor_unsmile"),
    ("KcELECTRA v2", "beomi/KcELECTRA-base-v2022"),
]

# 모델별 부정 라벨 매핑
MODEL_NEGATIVE_LABELS = {
    "matthewburke/korean_sentiment": ["LABEL_0"],
    "monologg/koelectra-small-finetuned-sentiment": ["negative"],
    "monologg/koelectra-base-finetuned-sentiment": ["negative"],
    "nlptown/bert-base-multilingual-uncased-sentiment": ["1 star", "2 stars"],
    "smilegate-ai/kor_unsmile": ["악플/욕설", "여성/가족", "남성", "성소수자", 
                                  "인종/국적", "연령", "지역", "종교", "기타 혐오"],
    "beomi/KcELECTRA-base-v2022": ["LABEL_1"],
}

# 9개 혐오 라벨 컬럼명
HATE_COLUMNS = ['여성/가족', '남성', '성소수자', '인종/국적', '연령', '지역', '종교', '기타 혐오', '악플/욕설']

def load_game_data():
    """데이터 로드 - 9개 혐오 라벨 중 하나라도 1이면 Abuse"""
    df = pd.read_csv(DATA_PATH, sep='\t')
    sentences = []
    labels = []
    
    for _, row in df.iterrows():
        text = str(row['문장']).strip()
        if not text: continue
        # 9개 혐오 라벨 중 하나라도 1이면 Abuse
        is_abuse = any(row.get(col, 0) == 1 for col in HATE_COLUMNS)
        sentences.append(text)
        labels.append(1 if is_abuse else 0)
    
    print(f"📂 데이터 로드: {len(sentences)}건")
    print(f"   Abuse (9개 혐오 라벨 중 1개 이상): {sum(labels)}")
    print(f"   Clean (정상): {len(labels) - sum(labels)}")
    return sentences, labels

sentences, true_labels = load_game_data()

## 2. 벤치마크 실행

In [ ]:
def benchmark_model(model_name, model_id, sentences, true_labels):
    print(f"\n{'='*60}\n🧪 {model_name}\n   {model_id}\n{'='*60}")
    
    result = {"model_name": model_name, "model_id": model_id, "error": None}
    
    try:
        # 모델 로드
        start = time.time()
        device = 0 if torch.cuda.is_available() else -1
        
        if "unsmile" in model_id.lower():
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            model = AutoModelForSequenceClassification.from_pretrained(model_id)
            model.eval()
            if torch.cuda.is_available(): model = model.cuda()
            classifier = None
        else:
            classifier = pipeline("sentiment-analysis", model=model_id, device=device)
            tokenizer = None; model = None
            
        result["load_time"] = time.time() - start
        print(f"✅ 로드 완료 ({result['load_time']:.2f}초)")
        
        # 예측
        predictions = []
        latencies = []
        
        for i, sentence in enumerate(sentences):
            if i % 50 == 0: print(f"   {i}/{len(sentences)}...")
            start = time.time()
            
            if "unsmile" in model_id.lower():
                inputs = tokenizer(sentence, return_tensors="pt", truncation=True, max_length=128)
                if torch.cuda.is_available():
                    inputs = {k: v.cuda() for k, v in inputs.items()}
                with torch.no_grad():
                    outputs = model(**inputs)
                    probs = torch.sigmoid(outputs.logits[0]).cpu().numpy()
                # 9개 혐오 라벨(인덱스 0~8) 중 하나라도 0.5 초과하면 Abuse
                hate_probs = probs[:9]
                is_abuse = np.any(hate_probs > 0.5)
            else:
                output = classifier(sentence)[0]
                neg_labels = MODEL_NEGATIVE_LABELS.get(model_id, [])
                is_abuse = output['label'] in neg_labels
            
            latencies.append((time.time() - start) * 1000)
            predictions.append(1 if is_abuse else 0)
            
        # 결과 계산
        result["avg_latency"] = np.mean(latencies)
        p, r, f1, _ = precision_recall_fscore_support(true_labels, predictions, labels=[0, 1], zero_division=0)
        
        result["clean_f1"] = f1[0]
        result["abuse_precision"] = p[1]
        result["abuse_recall"] = r[1]
        result["abuse_f1"] = f1[1]
        result["accuracy"] = np.mean(np.array(predictions) == np.array(true_labels))
        
        cm = confusion_matrix(true_labels, predictions)
        result["tn"], result["fp"], result["fn"], result["tp"] = cm.ravel()
        
        print(f"📊 Abuse Recall: {result['abuse_recall']:.2%}, F1: {result['abuse_f1']:.2%}")
        
    except Exception as e:
        result["error"] = str(e)
        print(f"❌ 에러: {e}")
        
    finally:
        if classifier: del classifier
        if model: del model
        gc.collect()
        torch.cuda.empty_cache()
        
    return result

all_results = []
for name, mid in MODELS_TO_TEST:
    all_results.append(benchmark_model(name, mid, sentences, true_labels))

## 3. 결과 시각화

In [ ]:
successful = [r for r in all_results if not r.get("error")]
names = [r["model_name"] for r in successful]
recalls = [r["abuse_recall"] * 100 for r in successful]
f1s = [r["abuse_f1"] * 100 for r in successful]

best_idx = np.argmax(recalls)

# 차트 그리기
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors = ['#2ECC71' if i == best_idx else '#95A5A6' for i in range(len(names))]

# Abuse Recall
ax = axes[0]
bars = ax.barh(names, recalls, color=colors, edgecolor='black')
ax.set_title(f'Abuse Recall (Best: {names[best_idx]})', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
for i, v in enumerate(recalls):
    ax.text(v + 1, i, f'{v:.1f}%', va='center', fontweight='bold' if i==best_idx else 'normal')

# Abuse F1
ax = axes[1]
bars = ax.barh(names, f1s, color=colors, edgecolor='black')
ax.set_title('Abuse F1 Score', fontsize=14, fontweight='bold')
ax.set_xlim(0, 100)
for i, v in enumerate(f1s):
    ax.text(v + 1, i, f'{v:.1f}%', va='center')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/benchmark_chart.png')
plt.show()

## 4. 결과 저장

In [ ]:
# CSV 저장
df_res = pd.DataFrame(successful)
cols = ['model_name', 'abuse_recall', 'abuse_f1', 'avg_latency', 'accuracy']
df_res = df_res[[c for c in cols if c in df_res.columns]]
df_res.to_csv(f'{RESULTS_DIR}/benchmark_results.csv', index=False)

# Markdown 생성
md = f"# 🎮 벤치마크 결과\n\n| 모델 | Abuse Recall | F1 Score |\n|---|---|---|\n"
for r in successful:
    md += f"| {r['model_name']} | {r['abuse_recall']*100:.2f}% | {r['abuse_f1']*100:.2f}% |\n"

with open(f'{RESULTS_DIR}/SUMMARY.md', 'w', encoding='utf-8') as f:
    f.write(md)

print("✅ 저장 완료!")